In [1]:
import awkward as ak
import numpy as np
import os
import json
import time
import uproot

from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from coffea.analysis_tools import PackedSelection

import sys
sys.path.append("..")
from src.processors.Processor import Processor
from src.processors.GenMatch import GenMatch

# Dataset

In [2]:
base = '/data/pubfs/fudawei/mc'
basedir = {d: os.path.join(base, d) for d in os.listdir(base) if d=='GJets' or d=='ZpToHGamma'}
basedir

FileNotFoundError: [Errno 2] No such file or directory: '/data/pubfs/fudawei/mc'

In [3]:
samples = {s: [] for s in basedir}
for s in basedir:
    for (current_path, dirs, files) in os.walk(basedir[s]):
        for f in files:
            if f.endswith('.root'):
                samples[s].append(os.path.join(current_path, f))
samples

{'GJets': ['/data/pubfs/fudawei/mc/GJets/HT-40To100/3B93385F-77FD-6B47-BF83-8476E32C7515.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/B7ED4838-B32C-E347-BD38-0FE7F990A127.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/42A269A9-5AB5-1045-B806-EE5D4588043B.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/5C460723-1125-3540-8FDE-472EF4ED064C.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/A7BEFB69-C329-C445-AA66-6758B5AD5CF4.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/59666184-B2AA-464C-8946-7384CB7F3626.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/7B14B631-E17C-FF41-B1B1-8AF5F9EE69D0.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/E4DB23B8-2ECA-2147-A8F1-BB5C1FCBCCC4.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/ED0DC548-72E5-D146-BD08-CC494A7BE30B.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/1D64B0C2-65B4-BE4D-9478-4ED80332AB41.root',
  '/data/pubfs/fudawei/mc/GJets/HT-40To100/070AD8C7-183E-EE41-BB0F-04B720E8FBAB.root',
  '/data/pubfs/fudawei/mc/GJets/HT

# Batch test

In [24]:
uproot.open.defaults["xrootd_handler"] = uproot.MultithreadedXRootDSource
_events = NanoEventsFactory.from_root(
    file='/eos/cms/store/data/Run2016D/SinglePhoton/NANOAOD/HIPM_UL2016_MiniAODv2_NanoAODv9-v2/2530000/A7ACEC8A-EE81-CB45-8A27-2508EB8318D4.root',
    treepath='Events', schemaclass=NanoAODSchema
).events()
_events
#p=Processor(outdir='./test/', mode='mc_2018_ZpToHgamma')
#p.process(_events)

<NanoEventsArray [<event 276361:287:546789033>, ... ] type='823725 * event'>

In [23]:
_events.Photon.electronVeto

<Array [[True, True], [True, ... [], [True]] type='1854856 * var * bool[paramete...'>

In [19]:
_events.HLT.fields

['AK8PFJet360_TrimMass30',
 'AK8PFHT700_TrimR0p1PT0p03Mass50',
 'AK8PFHT650_TrimR0p1PT0p03Mass50',
 'AK8PFHT600_TrimR0p1PT0p03Mass50_BTagCSV_p20',
 'CaloJet500_NoJetID',
 'Dimuon13_PsiPrime',
 'Dimuon13_Upsilon',
 'Dimuon20_Jpsi',
 'DoubleEle24_22_eta2p1_WPLoose_Gsf',
 'DoubleEle33_CaloIdL',
 'DoubleEle33_CaloIdL_MW',
 'DoubleEle33_CaloIdL_GsfTrkIdVL_MW',
 'DoubleEle33_CaloIdL_GsfTrkIdVL',
 'DoubleEle37_Ele27_CaloIdL_GsfTrkIdVL',
 'DoubleMediumIsoPFTau32_Trk1_eta2p1_Reg',
 'DoubleMediumIsoPFTau35_Trk1_eta2p1_Reg',
 'DoubleMediumIsoPFTau40_Trk1_eta2p1_Reg',
 'DoubleMu33NoFiltersNoVtx',
 'DoubleMu38NoFiltersNoVtx',
 'DoubleMu23NoFiltersNoVtxDisplaced',
 'DoubleMu28NoFiltersNoVtxDisplaced',
 'DoubleMu4_3_Bs',
 'DoubleMu4_3_Jpsi_Displaced',
 'DoubleMu4_JpsiTrk_Displaced',
 'DoubleMu4_LowMassNonResonantTrk_Displaced',
 'DoubleMu4_PsiPrimeTrk_Displaced',
 'Mu7p5_L2Mu2_Jpsi',
 'Mu7p5_L2Mu2_Upsilon',
 'Mu7p5_Track2_Jpsi',
 'Mu7p5_Track3p5_Jpsi',
 'Mu7p5_Track7_Jpsi',
 'Mu7p5_Track2_Upsilon',
 

In [10]:
ak.sum(ak.num(_events.Electron)>0)

15452

In [6]:
heaviest_jet = ak.flatten(p.object['AK8jet'][ak.argmax(p.object['AK8jet'].msoftdrop, axis=1, keepdims=True)], axis=1)
leading_photon = ak.flatten(p.object['photon'][ak.argmax(p.object['photon'].pt, axis=1, keepdims=True)], axis=1)
p.object['photon-jet'] = heaviest_jet + leading_photon
delta_phi = heaviest_jet.phi - leading_photon.phi
delta_phi = ak.min([delta_phi, 2 * np.pi - delta_phi], axis=0)
final_cut = p.pass_cut(cutName='photon-jet_delta_phi', cut=(delta_phi>2.9), final=True)

In [11]:
p.object['AK8jet'][ak.argmax(p.object['AK8jet'].msoftdrop, axis=1, keepdims=True)]

<FatJetArray [None, None, None, ... None, None, None] type='303 * option[var * ?...'>

In [4]:
p.cuts.names

['photon-jet_delta_phi']

In [3]:
g = GenMatch()
d=g.ZpToHGamma(_events)
d

{'ZpToHGamma': array([ True,  True,  True, ...,  True,  True,  True]),
 'ZpToH(WW)Gamma': array([False, False, False, ..., False, False, False]),
 'ZpToH(bb)Gamma': array([False,  True,  True, ..., False,  True,  True]),
 'ZpToH(ZZ)Gamma': array([False, False, False, ..., False, False, False]),
 'ZpToH(tautau)Gamma': array([ True, False, False, ..., False, False, False]),
 'ZpToH(gammagamma)Gamma': array([False, False, False, ..., False, False, False]),
 'gen_HWW_decay_mode': <Array [None, None, None, ... None, None, None] type='33000 * ?int64'>}

In [41]:
re = d['gen_HWW_decay_mode'][d['ZpToH(WW)Gamma']]

In [44]:
ak.sum(re==32)/len(re), ak.sum(re>16)/len(re)

(0.45201056648077487, 0.8982976225418257)

In [8]:
ak.sum(d['ZpToH(WW)Gamma'])

6814

In [9]:
g.gen_cuts.names

['N_Zp == 1',
 'Zp to H Gamma',
 'H to 2 childs',
 'H to WW',
 'H to bb',
 'H to ZZ',
 'H to tautau',
 'H to gammagamma',
 'WW to 4 childs']

In [39]:
ak.sum(g.gen_cuts.all('Zp to H Gamma', 'N_Zp == 1')), ak.sum(g.gen_cuts.all('Zp to H Gamma', 'H to WW', 'N_Zp == 1')), ak.sum(g.gen_cuts.all('H to WW'))

(32297, 6814, 6956)

In [18]:
t=_events.GenPart[  # shape: (event, H_childs)
    (abs(_events.GenPart.pdgId) == g.PDGID['W+']) &
    _events.GenPart.hasFlags(['fromHardProcess', 'isLastCopy'])
]

In [11]:
ak.sum(d['ZpToHGamma'])

0

In [87]:
ak.mean(ak.flatten(t[t.pdgId==22], axis=1).hasFlags(['fromHardProcess', 'isLastCopy']))

1.0

In [11]:
ak.max((_events.GenPart.pdgId == g.PDGID["Zp"]) & _events.GenPart.hasFlags(g.GEN_FLAGS), axis=1)

<Array [True, True, True, ... True, True, True] type='50000 * ?bool'>

# Parallel computing

In [11]:
cutflow = {}
t0 = time.time()

In [12]:
def parallelProcess(samples, name):
    global cutflow, t0
    cutflow[name] = processor.run_uproot_job(
        fileset={name: samples[name]},
        treename="Events",
        processor_instance=Processor(outdir=f'../output/{name}'),
        executor=processor.futures_executor,
        executor_args={"schema": NanoAODSchema, "workers": 48}, # running on $workers cpu cores
    )
    cutflow['time_'+name] = (time.time() - t0)/60
    with open('../output/cutflow.txt', 'w') as file:
        json.dump(cutflow, file)

parallelProcess(samples=samples, name='ZpToHGamma')
parallelProcess(samples=samples, name='GJets')
#parallelProcess(samples=samples, name='WJetsToQQ')
#parallelProcess(samples=samples, name='ZJetsToQQ')
#parallelProcess(samples=samples, name='QCD')



Processing:   0%|          | 0/50 [00:00<?, ?chunk/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1962: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1962: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  ) < distutils.version.LooseVersion("2.0.0"):
/home/pku/fudawei/.local/lib/python3.8/site-packages/awkward/operations/convert.py:1960: DeprecationWarn

Preprocessing:   0%|          | 0/147 [00:00<?, ?file/s]

Processing:   0%|          | 0/521 [00:00<?, ?chunk/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 302 jobs, will wait for 97 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 96 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 95 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 93 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:249: UserWarning: Early stop: cancelled 0 jobs, will wait for 92 running jobs to complete
  warnings.warn(
/home/pku/fudawei/.local/lib/python3.8/site-packages/